# Calibrated Popularity and history-aware calibration

This notebook adds the two re-ranking methods the proposal describes on top of the ALS baseline: **fixed Calibrated Popularity (CP)** and **history-aware CP**. It then evaluates every setting with UPD, Precision@10, Recall@10 and NDCG@10, broken down by overall, history bucket and user segment.

**Inputs:** the exported `top-100-candidates` and `top-10-baseline` Parquet tables from `als-baseline.ipynb`, plus `data/ml-1m/ratings.dat`. The training/test split is rebuilt with the same seed and hash as the baseline, then verified against the exported candidates.

**Method (proposal Section 2.2):** for each user, build the top-10 greedily. At each step add the candidate that maximizes `(1 - lambda_u) * Rel(L) - lambda_u * JS(p(u), q(L))`. CP uses `lambda_u = lambda`. History-aware CP uses `lambda_u = lambda * min(1, n_u / N0)`, where `n_u` is the user's number of training ratings.

**Design choice to confirm with the team:** greedy selection decides *which* 10 movies are shown. The final list is then ordered by the ALS rank, so calibration changes list membership but not the relative order of what remains. At `lambda = 0` this reproduces the baseline exactly, and the notebook asserts that.

Run the cells in order with the Scala 2.12.18 / Java 17 kernel. Everything after the Spark steps runs on the driver; the data is small (604,000 candidate rows).

In [4]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

import $ivy.$
import $ivy.$
import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

In [5]:
val spark = SparkSession.builder()
  .appName("CS 7265 - Calibrated Popularity")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

assert(spark.version == "3.5.9")
println(s"Spark ${spark.version}, Scala ${scala.util.Properties.versionNumberString}")

Spark 3.5.9, Scala 2.12.18


spark: SparkSession = org.apache.spark.sql.SparkSession@3b9fa144
import spark.implicits._

## 1. Configuration

Set `baselineRunId` to the timestamped folder name under `outputs/als-baseline/` that holds the exports you want to calibrate. The grids follow the proposal; `lambda = 0.0` is added as the unadjusted reference point.

In [6]:
val baselineRunId = "2026-11-12T07-10-03.780907Z"   // e.g. 2026-11-02T18-04-11.123456Z
val splitSeed = 42L
val trainingFraction = 0.8
val listSize = 10
val candidatePoolSize = 100

val lambdas = Seq(0.1, 0.3, 0.5, 0.7, 0.9)
val n0Values = Seq(50, 100, 200)

val shortCutoff = 50     // short:  n_u < 50
val longCutoff = 200     // medium: 50 <= n_u <= 200, long: n_u > 200

// Reference values from accuracy-evaluation.ipynb (single split, 6,012 evaluated users)
val reportedPrecision = 0.0567
val reportedRecall = 0.0391
val reportedNdcg = 0.0513

val ratingsPath = Seq("data/ml-1m/ratings.dat", "../data/ml-1m/ratings.dat")
  .find(p => new java.io.File(p).isFile)
  .getOrElse(throw new IllegalArgumentException("Run from the repository root or notebooks folder with data/ml-1m/ present."))
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile.getParentFile.getParentFile.getParentFile
val baselineRunDir = new java.io.File(repositoryRoot, s"outputs/als-baseline/$baselineRunId")
require(baselineRunDir.isDirectory, s"Baseline run folder not found: ${baselineRunDir.getPath}")

val calibrationRunId = java.time.Instant.now().toString.replace(":", "-")
val calibrationDir = new java.io.File(repositoryRoot, s"outputs/calibration/$calibrationRunId")
println(s"Baseline run:    ${baselineRunDir.getPath}")
println(s"Calibration run: ${calibrationDir.getPath}")

Baseline run:    /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/als-baseline/2026-11-12T07-10-03.780907Z
Calibration run: /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-12T07-20-31.673101Z


baselineRunId: String = "2026-11-12T07-10-03.780907Z"
splitSeed: Long = 42L
trainingFraction: Double = 0.8
listSize: Int = 10
candidatePoolSize: Int = 100
lambdas: Seq[Double] = List(0.1, 0.3, 0.5, 0.7, 0.9)
n0Values: Seq[Int] = List(50, 100, 200)
shortCutoff: Int = 50
longCutoff: Int = 200
reportedPrecision: Double = 0.0567
reportedRecall: Double = 0.0391
reportedNdcg: Double = 0.0513
ratingsPath: String = "../data/ml-1m/ratings.dat"
repositoryRoot: java.io.File = /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration
baselineRunDir: java.io.File = /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/als-baseline/2026-11-12T07-10-03.780907Z
calibrationRunId: String = "2026-11-12T07-20-31.673101Z"
calibrationDir: java.io.File = /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-12T07-20-31.673101Z

## 2. Rebuild the training/test split

This repeats the baseline's per-user split: `xxhash64(splitSeed, userId, movieId)` ordering, `floor(0.8 * n)` training rows per user. The hash depends on the column types, so `userId` and `movieId` are read as `Int` exactly as in the baseline. Section 3 checks the result against the exported candidates.

In [7]:
val ratingSchema = StructType(Seq(
  StructField("userId", IntegerType, false),
  StructField("movieId", IntegerType, false),
  StructField("rating", FloatType, false),
  StructField("timestamp", LongType, false)
))

val ratingsDF = spark.read
  .option("sep", "::")
  .schema(ratingSchema)
  .csv(ratingsPath)

val ratingCount = ratingsDF.count()
assert(ratingCount == 1000209L, s"Expected 1,000,209 ratings, found $ratingCount")

val splitRanked = ratingsDF
  .withColumn("userRatingCount", count("*").over(Window.partitionBy("userId")))
  .withColumn("shuffleKey", xxhash64(lit(splitSeed), col("userId"), col("movieId")))
  .withColumn("ratingPosition", row_number().over(
    Window.partitionBy("userId").orderBy(col("shuffleKey"), col("movieId"))))
  .withColumn("trainingCount", floor(col("userRatingCount") * trainingFraction))

val trainingRatings = splitRanked.filter(col("ratingPosition") <= col("trainingCount"))
  .select("userId", "movieId", "rating", "timestamp").cache()
val testRatings = splitRanked.filter(col("ratingPosition") > col("trainingCount"))
  .select("userId", "movieId", "rating", "timestamp").cache()

val trainingRatingCount = trainingRatings.count()
val testRatingCount = testRatings.count()
assert(trainingRatingCount + testRatingCount == ratingCount)
println(s"Training: $trainingRatingCount ratings, test: $testRatingCount ratings")

Training: 797758 ratings, test: 202451 ratings


ratingSchema: StructType = StructType(
  StructField("userId", IntegerType, false, {}),
  StructField("movieId", IntegerType, false, {}),
  StructField("rating", FloatType, false, {}),
  StructField("timestamp", LongType, false, {})
)
ratingsDF: DataFrame = [userId: int, movieId: int ... 2 more fields]
ratingCount: Long = 1000209L
splitRanked: DataFrame = [userId: int, movieId: int ... 6 more fields]
trainingRatings: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
testRatings: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
trainingRatingCount: Long = 797758L
testRatingCount: Long = 202451L

## 3. Load the baseline exports and verify they match this split

The exported candidates must contain no training pair, and the top-10 baseline must be exactly the first ten candidates. A mismatch means the split rebuilt here differs from the one used for ALS.

In [8]:
val candidates = spark.read.parquet(new java.io.File(baselineRunDir, "top-100-candidates").getPath)
  .select("userId", "movieId", "recommendationRank", "relevanceScore")
val baselineTop10 = spark.read.parquet(new java.io.File(baselineRunDir, "top-10-baseline").getPath)
  .select("userId", "movieId", "recommendationRank", "relevanceScore")

val trainingUsers = trainingRatings.select("userId").distinct()
val nUsers = trainingUsers.count()

val candidatesInTraining = candidates.join(
  trainingRatings.select("userId", "movieId"), Seq("userId", "movieId"), "inner").count()
val candidateRows = candidates.count()
val baselineFromCandidates = candidates.filter(col("recommendationRank") <= listSize)
val baselineMismatch = baselineTop10.exceptAll(baselineFromCandidates).count() +
  baselineFromCandidates.exceptAll(baselineTop10).count()

println(s"Users: $nUsers, candidate rows: $candidateRows")
println(s"Candidate pairs found in rebuilt training split: $candidatesInTraining")
println(s"Baseline vs first-10 candidate mismatches: $baselineMismatch")
assert(candidatesInTraining == 0L, "Split mismatch: candidates overlap the rebuilt training set.")
assert(candidateRows == nUsers * candidatePoolSize, "Every user must have 100 candidates.")
assert(baselineMismatch == 0L, "Exported baseline must equal the first 10 candidates.")

Users: 6040, candidate rows: 604000
Candidate pairs found in rebuilt training split: 0
Baseline vs first-10 candidate mismatches: 0


candidates: DataFrame = [userId: int, movieId: int ... 2 more fields]
baselineTop10: DataFrame = [userId: int, movieId: int ... 2 more fields]
trainingUsers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int]
nUsers: Long = 6040L
candidatesInTraining: Long = 0L
candidateRows: Long = 604000L
baselineFromCandidates: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
baselineMismatch: Long = 0L

## 4. Build the inputs the evaluators need

These are the tables `evaluateUPD` documents but the repo does not yet contain. Everything is derived from **training data only**.

- **Movie popularity groups:** movies ordered by training rating count (ties by movie ID). A movie is Head (`H`) while the share of ratings accumulated *before* it is under 20%, Tail (`T`) once that share reaches 80%, otherwise Mid (`M`).
- **User profiles `p(u)`:** each user's share of training rating *values* in H, M, T, following the proposal.
- **Segments:** users split into three equal groups by Head share (highest third is `blockbuster-focused`, lowest third is `niche-focused`).
- **History buckets:** short `n_u < 50`, medium `50-200`, long `> 200`, using training rating counts.

In [9]:
val movieCounts = trainingRatings.groupBy("movieId").agg(count("*").as("n"))
val totalTrainingRatings = trainingRatingCount.toDouble
val popularityOrder = Window.orderBy(col("n").desc, col("movieId").asc)

val movieGroups = movieCounts
  .withColumn("cumulativeBefore",
    coalesce(sum("n").over(popularityOrder.rowsBetween(Window.unboundedPreceding, -1)), lit(0L)))
  .withColumn("shareBefore", col("cumulativeBefore") / lit(totalTrainingRatings))
  .withColumn("popularityGroup",
    when(col("shareBefore") < 0.2, "H").when(col("shareBefore") >= 0.8, "T").otherwise("M"))
  .select("movieId", "popularityGroup", "n")
  .cache()

println("Movie groups (items and share of training ratings):")
movieGroups.groupBy("popularityGroup")
  .agg(count("*").as("movies"), (sum("n") / lit(totalTrainingRatings)).as("ratingShare"))
  .orderBy("popularityGroup").show()

val movieGroupsOut = movieGroups.select("movieId", "popularityGroup")

Movie groups (items and share of training ratings):
+---------------+------+-------------------+
|popularityGroup|movies|        ratingShare|
+---------------+------+-------------------+
|              H|   114|0.20112615605233666|
|              M|  1067| 0.5989723700671131|
|              T|  2504|0.19990147388055024|
+---------------+------+-------------------+



movieCounts: DataFrame = [movieId: int, n: bigint]
totalTrainingRatings: Double = 797758.0
popularityOrder: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@519013b7
movieGroups: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [movieId: int, popularityGroup: string ... 1 more field]
movieGroupsOut: DataFrame = [movieId: int, popularityGroup: string]

In [10]:
val ratingValue = col("rating").cast("double")
val profiles = trainingRatings.join(movieGroupsOut, "movieId")
  .groupBy("userId")
  .agg(
    sum(ratingValue).as("total"),
    sum(when(col("popularityGroup") === "H", ratingValue).otherwise(0.0)).as("h"),
    sum(when(col("popularityGroup") === "M", ratingValue).otherwise(0.0)).as("m"),
    sum(when(col("popularityGroup") === "T", ratingValue).otherwise(0.0)).as("t"))
  .select(col("userId"),
    (col("h") / col("total")).as("head"),
    (col("m") / col("total")).as("mid"),
    (col("t") / col("total")).as("tail"))
  .cache()

val segmentTile = ntile(3).over(Window.orderBy(col("head").desc, col("userId").asc))
val segments = profiles.withColumn("tile", segmentTile)
  .select(col("userId"),
    when(col("tile") === 1, "blockbuster-focused")
      .when(col("tile") === 2, "diverse")
      .otherwise("niche-focused").as("segment"))
  .cache()

val userHistory = trainingRatings.groupBy("userId").agg(count("*").as("nTrain"))
  .withColumn("historyBucket",
    when(col("nTrain") < shortCutoff, "short")
      .when(col("nTrain") <= longCutoff, "medium")
      .otherwise("long"))
  .cache()

assert(profiles.count() == nUsers && segments.count() == nUsers && userHistory.count() == nUsers)
println("Segments and history buckets:")
segments.groupBy("segment").count().orderBy("segment").show()
userHistory.groupBy("historyBucket").agg(count("*").as("users"), min("nTrain"), max("nTrain"))
  .orderBy("historyBucket").show()

Segments and history buckets:
+-------------------+-----+
|            segment|count|
+-------------------+-----+
|blockbuster-focused| 2014|
|            diverse| 2013|
|      niche-focused| 2013|
+-------------------+-----+

+-------------+-----+-----------+-----------+
|historyBucket|users|min(nTrain)|max(nTrain)|
+-------------+-----+-----------+-----------+
|         long| 1218|        201|       1851|
|       medium| 2617|         50|        200|
|        short| 2205|         16|         49|
+-------------+-----+-----------+-----------+



ratingValue: org.apache.spark.sql.Column = CAST(rating AS DOUBLE)
profiles: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, head: double ... 2 more fields]
segmentTile: org.apache.spark.sql.Column = ntile(3) OVER (ORDER BY head DESC NULLS LAST, userId ASC NULLS FIRST unspecifiedframe$())
segments: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, segment: string]
userHistory: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, nTrain: bigint ... 1 more field]

## 5. Calibration engine

Everything here runs on the driver with plain Scala arrays, so nothing is serialized to Spark workers. `Calib.js` is base-2 Jensen-Shannon divergence, not square-rooted, so it matches the UPD notebook and ranges over 0-1.

In [11]:
case class Cand(movieId: Int, rank: Int, rel: Double, g: Int)   // g: 0 = H, 1 = M, 2 = T

object Calib {
  private val ln2 = math.log(2.0)

  def js(p: Array[Double], q: Array[Double]): Double = {
    var s = 0.0
    var i = 0
    while (i < 3) {
      val m = (p(i) + q(i)) / 2.0
      if (p(i) > 0.0) s += 0.5 * p(i) * math.log(p(i) / m) / ln2
      if (q(i) > 0.0) s += 0.5 * q(i) * math.log(q(i) / m) / ln2
      i += 1
    }
    s
  }

  def mix(list: Array[Cand]): Array[Double] = {
    val c = Array(0.0, 0.0, 0.0)
    list.foreach(x => c(x.g) += 1.0)
    c.map(_ / list.length)
  }

  // cands must be sorted by ALS rank. Ties go to the better-ranked candidate.
  def rerank(cands: Array[Cand], p: Array[Double], lambda: Double, k: Int): Array[Cand] = {
    val used = new Array[Boolean](cands.length)
    val chosen = new Array[Cand](k)
    val counts = Array(0.0, 0.0, 0.0)
    val q = new Array[Double](3)
    var relSum = 0.0
    var t = 0
    while (t < k) {
      val size = (t + 1).toDouble
      var best = -1
      var bestObj = Double.NegativeInfinity
      var i = 0
      while (i < cands.length) {
        if (!used(i)) {
          val g = cands(i).g
          counts(g) += 1.0
          q(0) = counts(0) / size; q(1) = counts(1) / size; q(2) = counts(2) / size
          counts(g) -= 1.0
          val obj = (1.0 - lambda) * (relSum + cands(i).rel) - lambda * js(p, q)
          if (obj > bestObj) { bestObj = obj; best = i }
        }
        i += 1
      }
      used(best) = true
      chosen(t) = cands(best)
      relSum += cands(best).rel
      counts(cands(best).g) += 1.0
      t += 1
    }
    chosen.sortBy(_.rank)    // present in ALS order; calibration only changes membership
  }
}

// JS sanity checks
val pHead = Array(1.0, 0.0, 0.0); val pTail = Array(0.0, 0.0, 1.0); val pMix = Array(0.2, 0.5, 0.3)
assert(math.abs(Calib.js(pHead, pTail) - 1.0) < 1e-12)
assert(math.abs(Calib.js(pMix, pMix)) < 1e-12)
assert(math.abs(Calib.js(pMix, pHead) - Calib.js(pHead, pMix)) < 1e-12)
println("JS sanity checks passed.")

JS sanity checks passed.


defined class Cand
defined object Calib
pHead: Array[Double] = Array(1.0, 0.0, 0.0)
pTail: Array[Double] = Array(0.0, 0.0, 1.0)
pMix: Array[Double] = Array(0.2, 0.5, 0.3)

In [12]:
val groupIndex = Map("H" -> 0, "M" -> 1, "T" -> 2)

val candsByUser: Map[Int, Array[Cand]] = candidates.join(movieGroupsOut, "movieId")
  .select("userId", "movieId", "recommendationRank", "relevanceScore", "popularityGroup")
  .collect()
  .map(r => r.getInt(0) -> Cand(r.getInt(1), r.getInt(2), r.getDouble(3), groupIndex(r.getString(4))))
  .groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).sortBy(_.rank) }

val profileByUser: Map[Int, Array[Double]] = profiles.collect()
  .map(r => r.getInt(0) -> Array(r.getDouble(1), r.getDouble(2), r.getDouble(3))).toMap
val nTrainByUser: Map[Int, Long] = userHistory.select("userId", "nTrain").collect()
  .map(r => r.getInt(0) -> r.getLong(1)).toMap
val segmentByUser: Map[Int, String] = segments.collect().map(r => r.getInt(0) -> r.getString(1)).toMap
val bucketByUser: Map[Int, String] = userHistory.select("userId", "historyBucket").collect()
  .map(r => r.getInt(0) -> r.getString(1)).toMap

val likedByUser: Map[Int, Set[Int]] = testRatings.filter(col("rating") >= 4.0)
  .select("userId", "movieId").collect()
  .map(r => r.getInt(0) -> r.getInt(1)).groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).toSet }

assert(candsByUser.size == nUsers.toInt)
assert(candsByUser.values.forall(_.length == candidatePoolSize), "Every candidate movie must have a popularity label.")
println(s"Users with candidates: ${candsByUser.size}; users with a liked test movie: ${likedByUser.size}")

Users with candidates: 6040; users with a liked test movie: 6012


groupIndex: Map[String, Int] = Map("H" -> 0, "M" -> 1, "T" -> 2)
candsByUser: Map[Int, Array[Cand]] = Map(
  2163 -> Array(
    Cand(572, 1, 1.0, 2),
    Cand(318, 2, 0.48849375919138893, 0),
    Cand(1851, 3, 0.4313731557205782, 2),
    Cand(50, 4, 0.429920316103758, 0),
    Cand(3233, 5, 0.40162604228350113, 2),
    Cand(2905, 6, 0.39489069915906017, 2),
    Cand(527, 7, 0.3662823843641026, 0),
    Cand(3245, 8, 0.36542490668572963, 2),
    Cand(2762, 9, 0.3109741849783141, 0),
    Cand(989, 10, 0.3077100082333506, 2),
    Cand(2930, 11, 0.2976180187677715, 2),
    Cand(37, 12, 0.27924928913997943, 2),
    Cand(110, 13, 0.2721839006876537, 0),
    Cand(853, 14, 0.26170425022272725, 2),
    Cand(1704, 15, 0.2428432978214519, 0),
    Cand(593, 16, 0.24190437576798668, 0),
    Cand(1234, 17, 0.23891085041798035, 1),
    Cand(3897, 18, 0.21528984772747076, 1),
    Cand(2329, 19, 0.2150714772195815, 1),
    Cand(1262, 20, 0.2126519746704437, 1),
    Cand(3092, 21, 0.20275488266141725, 2),

## 6. Settings and per-user evaluation

A `Setting` is one method with its parameters. `userLambda` applies the history-aware weight. UPD per user is the Jensen-Shannon divergence between `p(u)` and the final list's mix. Relevance metrics use test movies rated 4 or higher, and only users with at least one such movie are scored, matching `accuracy-evaluation.ipynb`.

In [13]:
case class Setting(method: String, lambda: Double, n0: Option[Int]) {
  def name: String = (method, n0) match {
    case ("baseline", _) => "baseline"
    case ("cp", _)       => f"cp_l$lambda%.1f"
    case (_, Some(n))    => f"ha_n${n}_l$lambda%.1f"
    case _               => method
  }
  def userLambda(n: Long): Double = n0 match {
    case Some(n0v) => lambda * math.min(1.0, n.toDouble / n0v)
    case None      => lambda
  }
}

val settings: Seq[Setting] =
  Seq(Setting("baseline", 0.0, None)) ++
  lambdas.map(l => Setting("cp", l, None)) ++
  (for (n0 <- n0Values; l <- lambdas) yield Setting("ha", l, Some(n0)))

def runSetting(s: Setting): Map[Int, Array[Cand]] =
  candsByUser.map { case (u, cands) =>
    u -> Calib.rerank(cands, profileByUser(u), s.userLambda(nTrainByUser(u)), listSize)
  }

case class UserRec(userId: Int, segment: String, bucket: String, mismatch: Double,
                   scored: Boolean, precision: Double, recall: Double, ndcg: Double)

def evaluate(lists: Map[Int, Array[Cand]]): Seq[UserRec] =
  lists.toSeq.map { case (u, list) =>
    val mismatch = Calib.js(profileByUser(u), Calib.mix(list))
    likedByUser.get(u) match {
      case Some(liked) =>
        val hits = list.map(c => liked.contains(c.movieId))
        val nHits = hits.count(identity).toDouble
        val dcg = hits.zipWithIndex.collect { case (true, i) => 1.0 / (math.log(i + 2.0) / math.log(2.0)) }.sum
        val idcg = (0 until math.min(liked.size, listSize))
          .map(i => 1.0 / (math.log(i + 2.0) / math.log(2.0))).sum
        UserRec(u, segmentByUser(u), bucketByUser(u), mismatch, true,
          nHits / listSize, nHits / liked.size, dcg / idcg)
      case None =>
        UserRec(u, segmentByUser(u), bucketByUser(u), mismatch, false, 0.0, 0.0, 0.0)
    }
  }

defined class Setting
settings: Seq[Setting] = List(
  Setting("baseline", 0.0, None),
  Setting("cp", 0.1, None),
  Setting("cp", 0.3, None),
  Setting("cp", 0.5, None),
  Setting("cp", 0.7, None),
  Setting("cp", 0.9, None),
  Setting("ha", 0.1, Some(50)),
  Setting("ha", 0.3, Some(50)),
  Setting("ha", 0.5, Some(50)),
  Setting("ha", 0.7, Some(50)),
  Setting("ha", 0.9, Some(50)),
  Setting("ha", 0.1, Some(100)),
  Setting("ha", 0.3, Some(100)),
  Setting("ha", 0.5, Some(100)),
  Setting("ha", 0.7, Some(100)),
  Setting("ha", 0.9, Some(100)),
  Setting("ha", 0.1, Some(200)),
  Setting("ha", 0.3, Some(200)),
  Setting("ha", 0.5, Some(200)),
  Setting("ha", 0.7, Some(200)),
  Setting("ha", 0.9, Some(200))
)
defined function runSetting
defined class UserRec
defined function evaluate

## 7. Verify the engine before the sweep

1. At `lambda = 0` the engine must return the exported baseline, in the same order.
2. History-aware calibration with a very large `N0` makes every `lambda_u` tiny, so it must also return the baseline.
3. The baseline accuracy must match the numbers reported in `accuracy-evaluation.ipynb`.

In [14]:
val baselineLists = runSetting(settings.head)
val exportedBaselineByUser: Map[Int, Seq[Int]] = baselineTop10.collect()
  .map(r => (r.getInt(0), r.getInt(2), r.getInt(1))).groupBy(_._1)
  .map { case (u, xs) => u -> xs.sortBy(_._2).map(_._3).toSeq }

val baselineDiffs = baselineLists.count { case (u, l) => l.map(_.movieId).toSeq != exportedBaselineByUser(u) }
assert(baselineDiffs == 0, s"$baselineDiffs users differ between lambda=0 and the exported baseline.")

val hugeN0Diffs = runSetting(Setting("ha", 0.9, Some(1000000000)))
  .count { case (u, l) => l.map(_.movieId).toSeq != exportedBaselineByUser(u) }
assert(hugeN0Diffs == 0, s"$hugeN0Diffs users differ when N0 is huge.")

val baseRecs = evaluate(baselineLists).filter(_.scored)
def avgOf(xs: Seq[Double]) = if (xs.isEmpty) Double.NaN else xs.sum / xs.size
val bP = avgOf(baseRecs.map(_.precision)); val bR = avgOf(baseRecs.map(_.recall)); val bN = avgOf(baseRecs.map(_.ndcg))
println(f"Baseline here:     users=${baseRecs.size}  P@10=$bP%.4f  R@10=$bR%.4f  NDCG@10=$bN%.4f")
println(f"Reported by Monika: users=6012  P@10=$reportedPrecision%.4f  R@10=$reportedRecall%.4f  NDCG@10=$reportedNdcg%.4f")
assert(math.abs(bP - reportedPrecision) < 5e-4 && math.abs(bR - reportedRecall) < 5e-4 &&
  math.abs(bN - reportedNdcg) < 5e-4, "Baseline accuracy differs from the reported values.")
println("Engine checks passed.")

Baseline here:     users=6012  P@10=0.0567  R@10=0.0391  NDCG@10=0.0513
Reported by Monika: users=6012  P@10=0.0567  R@10=0.0391  NDCG@10=0.0513
Engine checks passed.


baselineLists: Map[Int, Array[Cand]] = Map(
  2163 -> Array(
    Cand(572, 1, 1.0, 2),
    Cand(318, 2, 0.48849375919138893, 0),
    Cand(1851, 3, 0.4313731557205782, 2),
    Cand(50, 4, 0.429920316103758, 0),
    Cand(3233, 5, 0.40162604228350113, 2),
    Cand(2905, 6, 0.39489069915906017, 2),
    Cand(527, 7, 0.3662823843641026, 0),
    Cand(3245, 8, 0.36542490668572963, 2),
    Cand(2762, 9, 0.3109741849783141, 0),
    Cand(989, 10, 0.3077100082333506, 2)
  ),
  645 -> Array(
    Cand(572, 1, 1.0, 2),
    Cand(3245, 2, 0.8655015067782819, 2),
    Cand(858, 3, 0.6396412152879071, 0),
    Cand(2905, 4, 0.6234728113797798, 2),
    Cand(2019, 5, 0.5438701007502122, 1),
    Cand(912, 6, 0.5073993027350195, 0),
    Cand(318, 7, 0.5036943566353924, 0),
    Cand(527, 8, 0.4980921884596702, 0),
    Cand(50, 9, 0.49048007165309737, 0),
    Cand(2309, 10, 0.4871338009549647, 2)
  ),
  892 -> Array(
    Cand(572, 1, 1.0, 2),
    Cand(3233, 2, 0.4540489847471079, 2),
    Cand(527, 3, 0.412381229

## 8. Run every setting

Each setting is re-ranked for all users, evaluated, and its top-10 is exported to Parquet in the schema `evaluateUPD` accepts. Results are aggregated for the whole cohort, each history bucket and each segment. `overallUPD` follows the README definition (equal-weight mean of the three segment means); the group rows use the plain mean mismatch within the group.

In [15]:
case class ResultRow(setting: String, method: String, lambda: Double, n0: Int,
                     groupType: String, group: String, users: Int, scoredUsers: Int,
                     upd: Double, precision: Double, recall: Double, ndcg: Double)

def summarize(s: Setting, groupType: String, group: String, recs: Seq[UserRec]): ResultRow = {
  val scored = recs.filter(_.scored)
  ResultRow(s.name, s.method, s.lambda, s.n0.getOrElse(0), groupType, group, recs.size, scored.size,
    avgOf(recs.map(_.mismatch)), avgOf(scored.map(_.precision)), avgOf(scored.map(_.recall)), avgOf(scored.map(_.ndcg)))
}

def officialOverallUPD(recs: Seq[UserRec]): Double =
  avgOf(recs.groupBy(_.segment).values.map(g => avgOf(g.map(_.mismatch))).toSeq)

calibrationDir.mkdirs()
val allRows = scala.collection.mutable.ArrayBuffer[ResultRow]()

settings.foreach { s =>
  val lists = runSetting(s)
  assert(lists.values.forall(l => l.length == listSize && l.map(_.movieId).distinct.length == listSize))
  val recs = evaluate(lists)

  allRows += summarize(s, "all", "all", recs).copy(upd = officialOverallUPD(recs))
  Seq("short", "medium", "long").foreach(b => allRows += summarize(s, "history", b, recs.filter(_.bucket == b)))
  Seq("blockbuster-focused", "diverse", "niche-focused").foreach(g =>
    allRows += summarize(s, "segment", g, recs.filter(_.segment == g)))

  lists.toSeq.flatMap { case (u, l) =>
    l.zipWithIndex.map { case (c, i) => (u, c.movieId, i + 1, c.rel) }
  }.toDF("userId", "movieId", "recommendationRank", "relevanceScore")
    .coalesce(1).write.mode("errorifexists")
    .parquet(new java.io.File(calibrationDir, s"top-10-${s.name}").getPath)

  val a = allRows.find(r => r.setting == s.name && r.groupType == "all").get
  println(f"${s.name}%-16s UPD=${a.upd}%.4f  P@10=${a.precision}%.4f  R@10=${a.recall}%.4f  NDCG@10=${a.ndcg}%.4f")
}

baseline         UPD=0.3267  P@10=0.0567  R@10=0.0391  NDCG@10=0.0513
cp_l0.1          UPD=0.3042  P@10=0.0579  R@10=0.0400  NDCG@10=0.0521
cp_l0.3          UPD=0.2504  P@10=0.0606  R@10=0.0414  NDCG@10=0.0540
cp_l0.5          UPD=0.1911  P@10=0.0646  R@10=0.0435  NDCG@10=0.0569
cp_l0.7          UPD=0.1220  P@10=0.0716  R@10=0.0473  NDCG@10=0.0627
cp_l0.9          UPD=0.0314  P@10=0.0843  R@10=0.0549  NDCG@10=0.0777
ha_n50_l0.1      UPD=0.3075  P@10=0.0578  R@10=0.0397  NDCG@10=0.0520
ha_n50_l0.3      UPD=0.2614  P@10=0.0604  R@10=0.0410  NDCG@10=0.0538
ha_n50_l0.5      UPD=0.2118  P@10=0.0643  R@10=0.0429  NDCG@10=0.0566
ha_n50_l0.7      UPD=0.1545  P@10=0.0708  R@10=0.0459  NDCG@10=0.0617
ha_n50_l0.9      UPD=0.0836  P@10=0.0824  R@10=0.0514  NDCG@10=0.0746
ha_n100_l0.1     UPD=0.3108  P@10=0.0575  R@10=0.0395  NDCG@10=0.0518
ha_n100_l0.3     UPD=0.2748  P@10=0.0602  R@10=0.0407  NDCG@10=0.0536
ha_n100_l0.5     UPD=0.2350  P@10=0.0635  R@10=0.0419  NDCG@10=0.0559
ha_n100_l0.7     UPD

defined class ResultRow
defined function summarize
defined function officialOverallUPD
res15_3: Boolean = true
allRows: collection.mutable.ArrayBuffer[ResultRow] = ArrayBuffer(
  ResultRow(
    "baseline",
    "baseline",
    0.0,
    0,
    "all",
    "all",
    6040,
    6012,
    0.32670063657554066,
    0.05666999334664026,
    0.03906158255710513,
    0.05125402403241676
  ),
  ResultRow(
    "baseline",
    "baseline",
    0.0,
    0,
    "history",
    "short",
    2205,
    2178,
    0.31857435094948494,
    0.024977043158861515,
    0.04730737836798452,
    0.03191739012140629
  ),
  ResultRow(
    "baseline",
    "baseline",
    0.0,
    0,
    "history",
    "medium",
    2617,
    2616,
    0.3332925532905689,
...

## 9. Save results and the evaluator inputs

`results.csv` feeds `analysis/analyze_results.py`. The Parquet inputs use the exact schemas in the UPD notebook's Section 9, so `evaluateUPD` can be run on any exported `top-10-*` folder as an independent check of the numbers above. Keep this run folder together; it identifies the split and method for every file.

In [16]:
val csvFile = new java.io.File(calibrationDir, "results.csv")
val out = new java.io.PrintWriter(csvFile)
out.println("setting,method,lambda,n0,groupType,group,users,scoredUsers,upd,precision,recall,ndcg")
allRows.foreach(r => out.println(Seq(r.setting, r.method, r.lambda, r.n0, r.groupType, r.group,
  r.users, r.scoredUsers, r.upd, r.precision, r.recall, r.ndcg).mkString(",")))
out.close()

val inputsDir = new java.io.File(calibrationDir, "inputs")
profiles.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir, "user-popularity-profiles").getPath)
movieGroupsOut.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir, "movie-popularity-groups").getPath)
segments.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir, "user-segments").getPath)
trainingUsers.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir, "evaluation-users").getPath)
userHistory.select("userId", "nTrain", "historyBucket").coalesce(1).write.mode("errorifexists")
  .parquet(new java.io.File(inputsDir, "user-history").getPath)

println(s"Wrote ${allRows.size} result rows to ${csvFile.getPath}")
println(s"Evaluator inputs in ${inputsDir.getPath}")
println(s"Split: seed=$splitSeed, fraction=$trainingFraction; baseline run: $baselineRunId")

Wrote 147 result rows to /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-12T07-20-31.673101Z/results.csv
Evaluator inputs in /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-12T07-20-31.673101Z/inputs
Split: seed=42, fraction=0.8; baseline run: 2026-11-12T07-10-03.780907Z


csvFile: java.io.File = /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-12T07-20-31.673101Z/results.csv
out: java.io.PrintWriter = java.io.PrintWriter@5e747310
inputsDir: java.io.File = /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-12T07-20-31.673101Z/inputs

## 10. Cross-check with the team's UPD evaluator (recommended)

Open `upd-evaluation.ipynb`, run its definition cells, then use the paths printed above in its Section 9, pointing `recommendations` at one exported list such as `top-10-cp_l0.5`. `overallScore` should equal the `upd` value in the `all` row of `results.csv` for that setting, within `1e-9`.

## 11. Stop Spark

In [17]:
spark.stop()
println("Spark session stopped.")

Spark session stopped.
